# Integração do dataset e bibliotecas

In [ ]:
!pip install anonymizedf

import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import hashlib
import os

path = kagglehub.dataset_download("preritsaxena/fraud-detection")

print("Dataset baixado em:", path)

arquivos = os.listdir(path)
print("Arquivos encontrados:", arquivos)

csv_path = os.path.join(path, arquivos[0])

df = pd.read_csv(csv_path)

print("\nFirst 5 records:")
display(df.head())


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Users\raulf\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'kagglehub'

# Verificar dados duplicados

In [ ]:
duplicados = df.duplicated(["user_id", "signup_time", "purchase_time", "purchase_value", "device_id", "source", "browser", "sex", "age", "ip_address", "is_fraud"]).sum()
display("A quantidade de dados duplicados é: ", duplicados)

# Verificar users e devices duplicados

In [ ]:
#Esta verificação é importante pois podem haver bots fazendo compras ao mesmo tempo usando o mesmo id de usuario e dispositivo

user_duplicado = df.duplicated(subset=['user_id']).sum()
print(f"Quantidade de usuarios duplicados: {user_duplicado}")

compras_suspeitas = df.duplicated(subset=['purchase_time','device_id']).sum()
print(f"Quantidadede de compras suspeitas: {compras_suspeitas}")


# Formatação de dados.

Verificar dados irrelevantes.

In [ ]:
filtro = (
    df['device_id'].str.contains(r'[^a-zA-Z0-9]', na=False) |
    df['source'].str.contains(r'[^a-zA-Z0-9]', na=False) |
    df['browser'].str.contains(r'[^a-zA-Z0-9]', na=False)|
    df['sex'].str.contains(r'[^a-zA-Z0-9]', na=False)
)
print(f"Quantidade de dados irrelevantes: {filtro}")
if filtro.sum() > 0:
    display(df[filtro].head())

Remover dados irrelevantes

In [ ]:
apagar = (
    df['device_id'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['source'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['browser'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['sex'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['age'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['user_id'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['signup_time'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['purchase_time'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['purchase_value'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['ip_address'].replace(r'[^a-zA-Z0-9]','',regex=True ),
    df['is_fraud'].replace(r'[^a-zA-Z0-9]','',regex=True )
)

print("Colunas limpas")

# Ocultar dados sensiveis

In [ ]:

# Ocultar dados sensíveis

# Anonimização da coluna 'sex': (mantido conforme solicitado)
df['sex'] = pd.factorize(df['sex'])[0] # Feminino = 1, Masculino = 0

# Anonimização da coluna 'age': Multiplicando a idade por 2
# Garantimos que a coluna 'age' seja numérica. Valores não numéricos (como 'Unknown' de execuções anteriores)
# serão convertidos para NaN e depois preenchidos com 0 antes da multiplicação, garantindo a operação.
df['age'] = pd.to_numeric(df['age'], errors='coerce').fillna(0).astype(int) * 2  # Idade multiplicada por 2 para ocultação

# Anonimização das colunas 'ip_address' e 'device_id': Utilizando um padrão de hash simplificado (primeiros 8 caracteres SHA-256)
# Esta técnica transforma os valores originais em um hash encurtado, tornando-o mais simples visualmente.
# Ainda permite rastrear se o mesmo IP ou Device ID original aparece novamente, mas com um padrão mais conciso.

def hash_value(value):
    if pd.isna(value):
        return None
    # Retorna os primeiros 8 caracteres do hash SHA-256 para um padrão mais simples
    return hashlib.sha256(str(value).encode()).hexdigest()[:8]

df['ip_address'] = df['ip_address'].apply(hash_value) # IP Address ocultado com hash SHA-256 (primeiros 8 caracteres)
df['device_id'] = df['device_id'].apply(hash_value)   # Device ID ocultado com hash SHA-256 (primeiros 8 caracteres)

print("Tabela com dados anonimizados:")
df.head()


## cálculos para extrair características dos dados e entender a distribuição deles.

Calculado media, mediaa e a moda

In [ ]:
print("Medidas de Localidade para 'purchase_value':")
mean_purchase_value = df['purchase_value'].mean()
median_purchase_value = df['purchase_value'].median()
mode_purchase_value = df['purchase_value'].mode()

print(f"  Média (purchase_value): {mean_purchase_value:.2f}")
print(f"  Mediana (purchase_value): {median_purchase_value:.2f}")
print(f"  Moda (purchase_value): {mode_purchase_value.tolist()}") # tolist() para exibir todos os modos se houver mais de um

print("\nMedidas de Localidade para 'age':")
mean_age = df['age'].mean()
median_age = df['age'].median()
mode_age = df['age'].mode()

print(f"  Média (age): {mean_age:.2f}")
print(f"  Mediana (age): {median_age:.2f}")
print(f"  Moda (age): {mode_age.tolist()}") # tolist() para exibir todos os modos se houver mais de um


Calculando medidas de espalhamento "Amplitude,Variante, Desvio Padrão"



In [ ]:
print("\nMedidas de Espalhamento para 'purchase_value':")
range_purchase_value = df['purchase_value'].max() - df['purchase_value'].min()
variance_purchase_value = df['purchase_value'].var()
std_dev_purchase_value = df['purchase_value'].std()

print(f"  Amplitude (purchase_value): {range_purchase_value:.2f}")
print(f"  Variância (purchase_value): {variance_purchase_value:.2f}")
print(f"  Desvio-Padrão (purchase_value): {std_dev_purchase_value:.2f}")

print("\nMedidas de Espalhamento para 'age':")
range_age = df['age'].max() - df['age'].min()
variance_age = df['age'].var()
std_dev_age = df['age'].std()

print(f"  Amplitude (age): {range_age:.2f}")
print(f"  Variância (age): {variance_age:.2f}")
print(f"  Desvio-Padrão (age): {std_dev_age:.2f}")


## Métricas (interpretação)

Para **purchase_value**, a **média** (~36,94) ficou um pouco acima da **mediana** (~35,00), o que sugere uma distribuição levemente assimétrica (com alguns valores mais altos puxando a média para cima). O **desvio padrão** (~18,32) indica uma variação considerável em torno do valor típico, ou seja, existem compras bem abaixo e bem acima do centro. Para **age**, a **média** (~132,56) e a **mediana** (~132,00) ficaram muito próximas, indicando uma distribuição mais “centrada”. O **desvio padrão** (~34,47) mostra que as idades estão relativamente espalhadas em torno do centro (lembrando que a coluna foi anonimizada multiplicando a idade por 2; para interpretar em “anos”, basta dividir por 2).

In [ ]:
# 2) Histograma de Idade (age foi anonimizada multiplicando por 2)
age_anos = pd.to_numeric(df['age'], errors='coerce') / 2
age_anos = age_anos.dropna()

bin_size = 5
start = int((age_anos.min() // bin_size) * bin_size)
end = int(((age_anos.max() // bin_size) + 1) * bin_size)
bins = list(range(start, end + bin_size, bin_size))

age_faixas = pd.cut(age_anos, bins=bins, right=False)
contagens = age_faixas.value_counts().sort_index()

faixa_top = contagens.idxmax()
qtd_top = int(contagens.max())

plt.figure(figsize=(10, 4))
plt.hist(age_anos, bins=bins, edgecolor='black')
plt.title('Distribuição de Idade (em anos; age/2)')
plt.xlabel('Idade (anos)')
plt.ylabel('Quantidade de usuários')
plt.xticks(bins, rotation=45)
plt.tight_layout()
plt.show()

print(f"Faixa etária com mais usuários: {faixa_top} (n={qtd_top})")


In [ ]:
# 3) Proporção de Fraudes
is_fraud_num = pd.to_numeric(df['is_fraud'], errors='coerce').fillna(0).astype(int)

contagens_fraude = is_fraud_num.value_counts().sort_index()
rotulos = ['Não fraude (0)', 'Fraude (1)']

plt.figure(figsize=(6, 4))
plt.bar(rotulos, [contagens_fraude.get(0, 0), contagens_fraude.get(1, 0)], edgecolor='black')
plt.title('Proporção de fraudes no dataset')
plt.ylabel('Quantidade de transações')
plt.tight_layout()
plt.show()

pct_fraude = is_fraud_num.mean() * 100
print(f"Porcentagem de fraudes no dataset: {pct_fraude:.2f}%")


In [ ]:
# 4) Uso de Tecnologia: browser e source mais comuns
browser_counts = df['browser'].astype(str).value_counts()
source_counts = df['source'].astype(str).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

browser_counts.plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_title('Browsers mais comuns')
axes[0].set_xlabel('Browser')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=45)

source_counts.plot(kind='bar', ax=axes[1], edgecolor='black')
axes[1].set_title('Sources mais comuns')
axes[1].set_xlabel('Source')
axes[1].set_ylabel('Quantidade')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"Browser mais comum: {browser_counts.index[0]} (n={int(browser_counts.iloc[0])})")
print(f"Source mais comum: {source_counts.index[0]} (n={int(source_counts.iloc[0])})")


## Interpretação das visualizações

Nesta etapa, os gráficos ajudam a investigar padrões que podem indicar fraude. Nos **boxplots**, os pontos fora do padrão esperado são tratados como outliers pelo critério IQR; compras muito acima do limite superior podem ser suspeitas, mas precisam ser comparadas com a proporção real de fraudes para evitar uma conclusão precipitada. No **gráfico de dispersão**, a idade é cruzada com o valor da compra e as fraudes aparecem em outra cor, permitindo observar se os golpes se concentram em faixas de idade ou valores específicos. Por fim, a **matriz de correlação** mostra a força da relação entre as colunas numéricas: valores próximos de 1 ou -1 indicam ligação mais forte, enquanto valores próximos de 0 indicam pouca relação com **is_fraud**.

In [ ]:
# 5) Caça aos Outliers (Boxplots)
purchase_value_num = pd.to_numeric(df['purchase_value'], errors='coerce')
age_anos = pd.to_numeric(df['age'], errors='coerce') / 2  # age foi anonimizada multiplicando por 2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(purchase_value_num.dropna())
axes[0].set_title('Boxplot de Valor da Compra')
axes[0].set_ylabel('Valor da compra')

axes[1].boxplot(age_anos.dropna())
axes[1].set_title('Boxplot de Idade')
axes[1].set_ylabel('Idade (anos)')

plt.tight_layout()
plt.show()

def calcular_outliers_iqr(serie):
    serie = serie.dropna()
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    outliers = serie[(serie < limite_inferior) | (serie > limite_superior)]
    return outliers, limite_inferior, limite_superior

outliers_compra, limite_inf_compra, limite_sup_compra = calcular_outliers_iqr(purchase_value_num)
outliers_idade, limite_inf_idade, limite_sup_idade = calcular_outliers_iqr(age_anos)

print(f"Outliers em purchase_value: {len(outliers_compra)}")
print(f"Limite superior para purchase_value: {limite_sup_compra:.2f}")

if len(outliers_compra) > 0:
    print(f"Maior compra encontrada: {outliers_compra.max():.2f}")

print(f"\nOutliers em age: {len(outliers_idade)}")
print(f"Limites para idade: {limite_inf_idade:.2f} até {limite_sup_idade:.2f} anos")

if len(outliers_compra) > 0:
    print("\nResposta: existem compras acima do padrão esperado pelo critério IQR. Elas podem ser investigadas como suspeitas, mas o valor alto sozinho não confirma fraude.")
else:
    print("\nResposta: não aparecem compras absurdamente altas pelo critério IQR.")

In [ ]:
# 6) Idade vs. Valor (Dispersão)
purchase_value_num = pd.to_numeric(df['purchase_value'], errors='coerce')
age_anos = pd.to_numeric(df['age'], errors='coerce') / 2
is_fraud_num = pd.to_numeric(df['is_fraud'], errors='coerce').fillna(0).astype(int)

df_dispersao = pd.DataFrame({
    'age_anos': age_anos,
    'purchase_value': purchase_value_num,
    'is_fraud': is_fraud_num
}).dropna()

plt.figure(figsize=(10, 5))

nao_fraude = df_dispersao[df_dispersao['is_fraud'] == 0]
fraude = df_dispersao[df_dispersao['is_fraud'] == 1]

plt.scatter(nao_fraude['age_anos'], nao_fraude['purchase_value'], alpha=0.25, s=18, label='Não fraude (0)')
plt.scatter(fraude['age_anos'], fraude['purchase_value'], alpha=0.75, s=24, label='Fraude (1)')

plt.title('Idade vs. Valor da Compra')
plt.xlabel('Idade (anos)')
plt.ylabel('Valor da compra')
plt.legend()
plt.tight_layout()
plt.show()

medias_por_classe = df_dispersao.groupby('is_fraud')[['age_anos', 'purchase_value']].mean()
medias_por_classe.index = medias_por_classe.index.map({0: 'Não fraude', 1: 'Fraude'})

display(medias_por_classe)

print("Resposta: os pontos de fraude mostram se existe concentração em valores de compra mais altos ou em alguma faixa etária específica.")

In [ ]:
# 7) Matriz de Correlação
df['purchase_value'] = pd.to_numeric(df['purchase_value'], errors='coerce')
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['is_fraud'] = pd.to_numeric(df['is_fraud'], errors='coerce')

correlacao = df.corr(numeric_only=True)

plt.figure(figsize=(10, 7))
mapa = plt.imshow(correlacao, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(mapa)

plt.xticks(range(len(correlacao.columns)), correlacao.columns, rotation=45, ha='right')
plt.yticks(range(len(correlacao.index)), correlacao.index)
plt.title('Heatmap da Matriz de Correlação')

for i in range(len(correlacao.index)):
    for j in range(len(correlacao.columns)):
        valor = correlacao.iloc[i, j]
        plt.text(j, i, f'{valor:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

display(correlacao)

if 'is_fraud' in correlacao.columns:
    correlacao_fraude = correlacao['is_fraud'].drop('is_fraud').sort_values(
        key=lambda serie: serie.abs(),
        ascending=False
    )

    display(correlacao_fraude.to_frame('correlacao_com_is_fraud'))

    coluna_mais_forte = correlacao_fraude.abs().idxmax()
    valor_mais_forte = correlacao_fraude[coluna_mais_forte]

    print(f"Resposta: a coluna com maior ligação com is_fraud é '{coluna_mais_forte}' com correlação de {valor_mais_forte:.2f}.")
else:
    print("A coluna is_fraud não está disponível como numérica para calcular a correlação.")